# Setup

## **Imports**

In [ ]:
!pip install -q segmentation-models-pytorch

In [18]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp

from scipy.ndimage import binary_propagation, label

from tqdm.auto import tqdm

from datetime import datetime


## **Paths and Device**

In [19]:
ROOT = Path(
    "/kaggle/input/competitions/filament-segmentation-2026/"
    "MAGFiLO_1.0_Kaggle_2026"
)

TRAIN_IMAGES = ROOT / "train" / "train_images"

ANNOTATIONS_PATH = (
    ROOT
    / "train"
    / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
)

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

gpu_count = torch.cuda.device_count()

print("Device:", device)
print("Available GPUs:", gpu_count)

if torch.cuda.is_available():
    for i in range(gpu_count):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

Device: cuda
Available GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## **Model Utilities**

In [20]:
def prepare_model(model):
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    return model.to(device)

# Data
## Load Annotations

In [21]:
with open(ANNOTATIONS_PATH, "r") as f:
    annotations_data = json.load(f)

## Data Utilities

In [22]:
def index_by(items, key):
    index = {}

    for item in items:
        value = item[key]
        index.setdefault(value, []).append(item)

    return index


image_entries_by_filename = index_by(
    annotations_data["images"],
    "file_name"
)

annotations_by_image_id = index_by(
    annotations_data["annotations"],
    "image_id"
)


def load_image(filename):
    return Image.open(
        TRAIN_IMAGES / filename
    ).convert("L")


def get_annotated_filenames():
    return sorted(
        image_entries_by_filename.keys()
    )


def get_image_entries(filename):
    return image_entries_by_filename.get(
        filename,
        []
    )


def get_annotations(image_id):
    return annotations_by_image_id.get(
        image_id,
        []
    )


def get_image_annotations(
    filename,
    annotation_set=0
):
    image_entries = get_image_entries(filename)

    if not image_entries:
        raise ValueError(
            f"No annotations found for {filename}"
        )

    if not 0 <= annotation_set < len(image_entries):
        raise ValueError(
            f"Annotation set {annotation_set} does not exist "
            f"for {filename}. "
            f"Available sets: {len(image_entries)}"
        )

    image_id = image_entries[annotation_set]["id"]

    return get_annotations(image_id)


def annotations_to_mask(
    annotations,
    image_size
):
    mask = Image.new(
        "L",
        image_size,
        0
    )

    draw = ImageDraw.Draw(mask)

    for annotation in annotations:
        segmentation = annotation["segmentation"][0]

        points = [
            (
                segmentation[i],
                segmentation[i + 1]
            )
            for i in range(
                0,
                len(segmentation),
                2
            )
        ]

        draw.polygon(
            points,
            fill=1
        )

    return np.array(
        mask,
        dtype=np.uint8
    )


def get_image_and_mask(
    filename,
    annotation_set=0
):
    image = load_image(filename)

    annotations = get_image_annotations(
        filename,
        annotation_set
    )

    mask = annotations_to_mask(
        annotations,
        image.size
    )

    return image, mask


def get_training_samples():
    samples = []

    for filename, image_entries in (
        image_entries_by_filename.items()
    ):
        for annotation_set in range(
            len(image_entries)
        ):
            samples.append(
                (filename, annotation_set)
            )

    return sorted(samples)

## Dataset

In [23]:
class FilamentDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        filename, annotation_set = self.samples[index]

        image, mask = get_image_and_mask(
            filename,
            annotation_set
        )

        image = np.array(
            image,
            dtype=np.float32
        ) / 255.0

        mask = mask.astype(np.float32)

        image = torch.from_numpy(
            image
        ).unsqueeze(0)

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)

        return image, mask

## Data Splitting

**Splitting Utilities**

In [24]:
def get_observation_date(filename):
    timestamp = filename[:14]

    return datetime.strptime(
        timestamp,
        "%Y%m%d%H%M%S"
    ).date()


def group_filenames_by_date(filenames):
    filenames_by_date = {}

    for filename in filenames:
        date = get_observation_date(filename)

        filenames_by_date.setdefault(
            date,
            []
        ).append(filename)

    return filenames_by_date


def split_filenames_by_date(
    filenames,
    validation_fraction=0.2,
    seed=42
):
    filenames_by_date = group_filenames_by_date(
        filenames
    )

    dates = np.array(
        sorted(filenames_by_date.keys())
    )

    rng = np.random.default_rng(seed)
    rng.shuffle(dates)

    validation_count = max(
        1,
        round(len(dates) * validation_fraction)
    )

    validation_dates = set(
        dates[:validation_count]
    )

    train_filenames = []
    validation_filenames = []

    for date, date_filenames in filenames_by_date.items():
        if date in validation_dates:
            validation_filenames.extend(date_filenames)
        else:
            train_filenames.extend(date_filenames)

    return (
        sorted(train_filenames),
        sorted(validation_filenames)
    )


def get_samples_for_filenames(filenames):
    samples = []

    for filename in filenames:
        image_entries = get_image_entries(filename)

        for annotation_set in range(
            len(image_entries)
        ):
            samples.append(
                (filename, annotation_set)
            )

    return samples

**Split Script**

In [25]:
filenames = get_annotated_filenames()

train_filenames, validation_filenames = (
    split_filenames_by_date(filenames)
)

train_samples = get_samples_for_filenames(
    train_filenames
)

validation_samples = get_samples_for_filenames(
    validation_filenames
)

print("Training images:", len(train_filenames))
print("Validation images:", len(validation_filenames))

print("Training samples:", len(train_samples))
print("Validation samples:", len(validation_samples))

Training images: 562
Validation images: 145
Training samples: 905
Validation samples: 249


## DataLoaders

In [26]:
train_dataset = FilamentDataset(train_samples)
validation_dataset = FilamentDataset(validation_samples)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

Train batches: 452
Validation batches: 125
Image batch shape: torch.Size([2, 1, 2048, 2048])
Mask batch shape: torch.Size([2, 1, 2048, 2048])


# Training
## Losses

In [27]:
def dice_loss(
    logits,
    targets,
    eps=1e-6
):
    probabilities = torch.sigmoid(
        logits
    )

    intersection = (
        probabilities * targets
    ).sum(dim=(1, 2, 3))

    total = (
        probabilities.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    dice = (
        2 * intersection + eps
    ) / (
        total + eps
    )

    return 1 - dice.mean()


def focal_loss(
    logits,
    targets,
    alpha=0.25,
    gamma=2.0
):
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none"
    )

    probabilities = torch.sigmoid(
        logits
    )

    p_t = (
        probabilities * targets
        + (1 - probabilities)
        * (1 - targets)
    )

    focal_weight = (
        1 - p_t
    ) ** gamma

    alpha_weight = (
        alpha * targets
        + (1 - alpha)
        * (1 - targets)
    )

    focal = (
        alpha_weight
        * focal_weight
        * bce
    )

    return focal.mean()


def segmentation_loss(
    logits,
    targets
):
    focal = focal_loss(
        logits,
        targets,
        alpha=0.25,
        gamma=2.0
    )

    dice = dice_loss(
        logits,
        targets
    )

    return focal + dice

## Metrics

In [28]:
def dice_score(
    logits,
    targets,
    threshold=0.5,
    eps=1e-6
):
    probabilities = torch.sigmoid(logits)
    predictions = (
        probabilities > threshold
    ).float()

    intersection = (
        predictions * targets
    ).sum(dim=(1, 2, 3))

    total = (
        predictions.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    dice = (
        2 * intersection + eps
    ) / (
        total + eps
    )

    return dice.mean().item()


def iou_score(
    logits,
    targets,
    threshold=0.5,
    eps=1e-6
):
    probabilities = torch.sigmoid(logits)
    predictions = (
        probabilities > threshold
    ).float()

    intersection = (
        predictions * targets
    ).sum(dim=(1, 2, 3))

    union = (
        predictions.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
        - intersection
    )

    iou = (
        intersection + eps
    ) / (
        union + eps
    )

    return iou.mean().item()


def extract_predicted_instances(
    probability_map,
    low_threshold=0.4,
    high_threshold=0.6,
    min_size=50
):
    low_mask = probability_map >= low_threshold
    high_mask = probability_map >= high_threshold

    grown_mask = binary_propagation(
        high_mask,
        mask=low_mask
    )

    labeled_mask, instance_count = label(
        grown_mask,
        structure=np.ones(
            (3, 3),
            dtype=np.uint8
        )
    )

    instances = []

    for instance_id in range(
        1,
        instance_count + 1
    ):
        instance_mask = (
            labeled_mask == instance_id
        )

        if instance_mask.sum() >= min_size:
            instances.append(instance_mask)

    return instances


def get_ground_truth_instances(
    filename,
    annotation_set
):
    annotations = get_image_annotations(
        filename,
        annotation_set
    )

    image = load_image(filename)
    instances = []

    for annotation in annotations:
        mask = Image.new(
            "L",
            image.size,
            0
        )

        draw = ImageDraw.Draw(mask)

        segmentation = annotation[
            "segmentation"
        ][0]

        points = [
            (
                segmentation[i],
                segmentation[i + 1]
            )
            for i in range(
                0,
                len(segmentation),
                2
            )
        ]

        draw.polygon(
            points,
            fill=1
        )

        instances.append(
            np.array(
                mask,
                dtype=bool
            )
        )

    return instances


def instance_iou(
    ground_truth,
    prediction
):
    intersection = np.logical_and(
        ground_truth,
        prediction
    ).sum()

    union = np.logical_or(
        ground_truth,
        prediction
    ).sum()

    if union == 0:
        return 0.0

    return intersection / union


def panoptic_quality(
    ground_truth_instances,
    predicted_instances,
    match_threshold=0.5
):
    iou_pairs = []

    for gt_index, ground_truth in enumerate(
        ground_truth_instances
    ):
        for pred_index, prediction in enumerate(
            predicted_instances
        ):
            iou = instance_iou(
                ground_truth,
                prediction
            )

            if iou > match_threshold:
                iou_pairs.append(
                    (
                        iou,
                        gt_index,
                        pred_index
                    )
                )

    iou_pairs.sort(
        reverse=True
    )

    matched_ground_truth = set()
    matched_predictions = set()
    matched_ious = []

    for iou, gt_index, pred_index in iou_pairs:
        if gt_index in matched_ground_truth:
            continue

        if pred_index in matched_predictions:
            continue

        matched_ground_truth.add(
            gt_index
        )

        matched_predictions.add(
            pred_index
        )

        matched_ious.append(
            iou
        )

    true_positives = len(
        matched_ious
    )

    false_positives = (
        len(predicted_instances)
        - true_positives
    )

    false_negatives = (
        len(ground_truth_instances)
        - true_positives
    )

    denominator = (
        true_positives
        + 0.5 * false_positives
        + 0.5 * false_negatives
    )

    if true_positives > 0:
        segmentation_quality = (
            sum(matched_ious)
            / true_positives
        )
    else:
        segmentation_quality = 0.0

    if denominator > 0:
        recognition_quality = (
            true_positives
            / denominator
        )

        pq = (
            sum(matched_ious)
            / denominator
        )
    else:
        recognition_quality = 1.0
        pq = 1.0

    return {
        "pq": pq,
        "sq": segmentation_quality,
        "rq": recognition_quality,
        "tp": true_positives,
        "fp": false_positives,
        "fn": false_negatives
    }


def count_instance_relations(
    ground_truth_instances,
    predicted_instances
):
    ground_truth_overlaps = [
        0
        for _ in ground_truth_instances
    ]

    prediction_overlaps = [
        0
        for _ in predicted_instances
    ]

    for gt_index, ground_truth in enumerate(
        ground_truth_instances
    ):
        for pred_index, prediction in enumerate(
            predicted_instances
        ):
            overlap = np.logical_and(
                ground_truth,
                prediction
            ).any()

            if overlap:
                ground_truth_overlaps[
                    gt_index
                ] += 1

                prediction_overlaps[
                    pred_index
                ] += 1

    one_to_many = sum(
        count > 1
        for count in ground_truth_overlaps
    )

    many_to_one = sum(
        count > 1
        for count in prediction_overlaps
    )

    return {
        "one_to_many": one_to_many,
        "many_to_one": many_to_one
    }


def instance_metrics(
    probability_map,
    filename,
    annotation_set
):
    predicted_instances = (
        extract_predicted_instances(
            probability_map
        )
    )

    ground_truth_instances = (
        get_ground_truth_instances(
            filename,
            annotation_set
        )
    )

    metrics = panoptic_quality(
        ground_truth_instances,
        predicted_instances
    )

    metrics.update(
        count_instance_relations(
            ground_truth_instances,
            predicted_instances
        )
    )

    return metrics

## Training Utilities

In [29]:
def save_training_history(
    history,
    checkpoint_path
):
    history_df = pd.DataFrame(history)

    history_df.insert(
        0,
        "epoch",
        range(1, len(history_df) + 1)
    )

    history_path = checkpoint_path.with_suffix(".csv")

    history_df.to_csv(
        history_path,
        index=False
    )

    print(
        f"Saved training history to: "
        f"{history_path}"
    )


def print_validation_metrics(
    train_loss,
    validation
):
    print(
        f"Train Loss:      {train_loss:.4f}"
    )

    print(
        f"Validation Loss: {validation['loss']:.4f}"
    )

    print(
        f"Dice:            {validation['dice']:.4f}"
    )

    print(
        f"IoU:             {validation['iou']:.4f}"
    )

    print()

    print(
        f"PQ:              {validation['pq']:.4f}"
    )

    print(
        f"├─ SQ:           {validation['sq']:.4f}"
    )

    print(
        f"└─ RQ:           {validation['rq']:.4f}"
    )

    print()

    print("Instances")

    print(
        f"├─ TP:           {validation['tp']}"
    )

    print(
        f"├─ FP:           {validation['fp']}"
    )

    print(
        f"├─ FN:           {validation['fn']}"
    )

    print(
        f"├─ One-to-many:  "
        f"{validation['one_to_many']}"
    )

    print(
        f"└─ Many-to-one:  "
        f"{validation['many_to_one']}"
    )


def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    scaler
):
    model.train()

    total_loss = 0.0

    progress_bar = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for images, masks in progress_bar:
        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = model(images)

            loss = criterion(
                logits,
                masks
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return total_loss / len(loader)


@torch.no_grad()
def validate_one_epoch(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    total_samples = 0

    total_tp = 0
    total_fp = 0
    total_fn = 0

    total_matched_iou = 0.0

    total_one_to_many = 0
    total_many_to_one = 0

    progress_bar = tqdm(
        loader,
        desc="Validation",
        leave=False
    )

    sample_index = 0

    for images, masks in progress_bar:
        batch_size = images.shape[0]

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = model(images)

            loss = criterion(
                logits,
                masks
            )

        dice = dice_score(
            logits,
            masks
        )

        iou = iou_score(
            logits,
            masks
        )

        total_loss += (
            loss.item() * batch_size
        )

        total_dice += (
            dice * batch_size
        )

        total_iou += (
            iou * batch_size
        )

        probabilities = torch.sigmoid(
            logits
        ).float().cpu().numpy()

        for batch_offset in range(
            batch_size
        ):
            filename, annotation_set = (
                loader.dataset.samples[
                    sample_index
                ]
            )

            probability_map = probabilities[
                batch_offset,
                0
            ]

            metrics = instance_metrics(
                probability_map,
                filename,
                annotation_set
            )

            tp = metrics["tp"]
            fp = metrics["fp"]
            fn = metrics["fn"]

            denominator = (
                tp
                + 0.5 * fp
                + 0.5 * fn
            )

            matched_iou = (
                metrics["pq"]
                * denominator
            )

            total_tp += tp
            total_fp += fp
            total_fn += fn

            total_matched_iou += (
                matched_iou
            )

            total_one_to_many += (
                metrics["one_to_many"]
            )

            total_many_to_one += (
                metrics["many_to_one"]
            )

            sample_index += 1

        total_samples += batch_size

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            dice=f"{dice:.4f}",
            iou=f"{iou:.4f}"
        )

    pq_denominator = (
        total_tp
        + 0.5 * total_fp
        + 0.5 * total_fn
    )

    if total_tp > 0:
        sq = (
            total_matched_iou
            / total_tp
        )
    else:
        sq = 0.0

    if pq_denominator > 0:
        rq = (
            total_tp
            / pq_denominator
        )

        pq = (
            total_matched_iou
            / pq_denominator
        )
    else:
        rq = 1.0
        pq = 1.0

    return {
        "loss": total_loss / total_samples,
        "dice": total_dice / total_samples,
        "iou": total_iou / total_samples,
        "pq": pq,
        "sq": sq,
        "rq": rq,
        "tp": total_tp,
        "fp": total_fp,
        "fn": total_fn,
        "one_to_many": total_one_to_many,
        "many_to_one": total_many_to_one
    }


def train_model(
    model,
    train_loader,
    validation_loader,
    optimizer,
    criterion,
    device,
    epochs,
    checkpoint_path,
    patience=3
):
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda")
    )

    history = {
        "train_loss": [],
        "validation_loss": [],
        "dice": [],
        "iou": [],
        "pq": [],
        "sq": [],
        "rq": [],
        "tp": [],
        "fp": [],
        "fn": [],
        "one_to_many": [],
        "many_to_one": []
    }

    best_pq = -float("inf")
    epochs_without_improvement = 0

    start_time = time.time()

    for epoch in range(epochs):
        print(
            f"\nEpoch {epoch + 1}/{epochs}"
        )

        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            scaler=scaler
        )

        validation = validate_one_epoch(
            model=model,
            loader=validation_loader,
            criterion=criterion,
            device=device
        )

        history["train_loss"].append(
            train_loss
        )

        history["validation_loss"].append(
            validation["loss"]
        )

        for metric in [
            "dice",
            "iou",
            "pq",
            "sq",
            "rq",
            "tp",
            "fp",
            "fn",
            "one_to_many",
            "many_to_one"
        ]:
            history[metric].append(
                validation[metric]
            )

        print_validation_metrics(
            train_loss,
            validation
        )

        if validation["pq"] > best_pq:
            best_pq = validation["pq"]
            epochs_without_improvement = 0

            model_to_save = (
                model.module
                if isinstance(
                    model,
                    nn.DataParallel
                )
                else model
            )

            torch.save(
                model_to_save.state_dict(),
                checkpoint_path
            )

            print(
                "Saved new best model "
                f"(PQ: {best_pq:.4f})."
            )

        else:
            epochs_without_improvement += 1

            print(
                f"No PQ improvement "
                f"({epochs_without_improvement}/"
                f"{patience})"
            )

        if (
            epochs_without_improvement
            >= patience
        ):
            print("Early stopping.")
            break

    elapsed = (
        time.time() - start_time
    )

    print(
        f"Training time: "
        f"{elapsed / 60:.1f} minutes"
    )

    save_training_history(
        history=history,
        checkpoint_path=checkpoint_path
    )

    return history

# Backbone

## U-Net++

In [32]:
def make_unetplusplus():
    return smp.UnetPlusPlus(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=1,
        classes=1
    )

# Experiments
## Experiment Utilities

In [33]:
def run_experiment(
    name,
    model,
    epochs=20,
    learning_rate=1e-4,
    patience=3
):
    checkpoint_path = (
        CHECKPOINT_DIR / f"{name}.pt"
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    parameter_count = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    model = prepare_model(model)

    print(f"Experiment: {name}")
    print(f"Parameters: {parameter_count:,}")
    print(f"Checkpoint: {checkpoint_path}")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate
    )

    history = train_model(
        model=model,
        train_loader=train_loader,
        validation_loader=validation_loader,
        optimizer=optimizer,
        criterion=segmentation_loss,
        device=device,
        epochs=epochs,
        checkpoint_path=checkpoint_path,
        patience=patience
    )

    return {
        "name": name,
        "history": history,
        "checkpoint_path": checkpoint_path,
        "parameters": parameter_count
    }

## U-Net++ Baseline

In [1]:
unetplusplus_focal_result = run_experiment(
    name="unetplusplus_resnet34_focal_dice",
    model=make_unetplusplus(),
    epochs=10,
    learning_rate=1e-4,
    patience=3
)

NameError: name 'run_experiment' is not defined